In [ ]:
from openai import OpenAI


client=OpenAI(api_key="")

In [16]:
training_file_small_all ='../Dataset/training_polymer_both_for_gpt.jsonl'
validation_file_small_all = '../Dataset/validation_polymer_both_for_gpt.jsonl'

In [17]:
import os


if os.path.exists(validation_file_small_all):
    print("File exists.")
else:
    print("File does not exist.")

File exists.


In [18]:
training_file_small_all=client.files.create(file=open(training_file_small_all,'rb'),
                    purpose='fine-tune')

validation_file_small_all=client.files.create(file=open(validation_file_small_all,'rb'),
                    purpose='fine-tune')
print(training_file_small_all)
print('---------------------------------')
print(validation_file_small_all)

FileObject(id='file-17AEPM3E4k23BmmzqdC92M', bytes=6455515, created_at=1754503564, filename='training_polymer_both_for_gpt.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)
---------------------------------
FileObject(id='file-DPRLj95EnyS2TyotkGzmZb', bytes=764765, created_at=1754503565, filename='validation_polymer_both_for_gpt.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)


In [25]:
response_small_all=client.fine_tuning.jobs.create(
    training_file=training_file_small_all.id,
    validation_file=validation_file_small_all.id,
    model='gpt-4o-mini-2024-07-18',
    hyperparameters={
    'n_epochs':'auto',
    'batch_size':'auto',  
    'learning_rate_multiplier': 0.1,
},
seed=42,
suffix="tsmp_monomers_v1"
)

print(response_small_all)

FineTuningJob(id='ftjob-qRGTbJ8M2u97I33ZPrauBcKz', created_at=1754504419, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size='auto', learning_rate_multiplier=0.1, n_epochs='auto'), model='gpt-4o-mini-2024-07-18', object='fine_tuning.job', organization_id='org-AVOdG0yGREZO41elnVjY6u7y', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-17AEPM3E4k23BmmzqdC92M', validation_file='file-DPRLj95EnyS2TyotkGzmZb', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size='auto', learning_rate_multiplier=0.1, n_epochs='auto'))), user_provided_suffix='tsmp_monomers_v1', usage_metrics=None, shared_with_openai=False, eval_id=None)


In [38]:
ft_model_small_all=client.fine_tuning.jobs.retrieve(response_small_all.id)#(response.id)
print(ft_model_small_all)

FineTuningJob(id='ftjob-qRGTbJ8M2u97I33ZPrauBcKz', created_at=1754504419, error=Error(code=None, message=None, param=None), fine_tuned_model='ft:gpt-4o-mini-2024-07-18:personal:tsmp-monomers-v1:C1dbxURY', finished_at=1754506627, hyperparameters=Hyperparameters(batch_size=7, learning_rate_multiplier=0.1, n_epochs=3), model='gpt-4o-mini-2024-07-18', object='fine_tuning.job', organization_id='org-AVOdG0yGREZO41elnVjY6u7y', result_files=['file-BRBLFthK4iaju4ovm9465H'], seed=42, status='succeeded', trained_tokens=4553622, training_file='file-17AEPM3E4k23BmmzqdC92M', validation_file='file-DPRLj95EnyS2TyotkGzmZb', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=7, learning_rate_multiplier=0.1, n_epochs=3))), user_provided_suffix='tsmp_monomers_v1', usage_metrics=None, shared_with_openai=False, eval_id=None)


In [3]:
ft_model_small_all=client.fine_tuning.jobs.retrieve('ftjob-qRGTbJ8M2u97I33ZPrauBcKz')#(response.id)
print(ft_model_small_all)

FineTuningJob(id='ftjob-qRGTbJ8M2u97I33ZPrauBcKz', created_at=1754504419, error=Error(code=None, message=None, param=None), fine_tuned_model='ft:gpt-4o-mini-2024-07-18:personal:tsmp-monomers-v1:C1dbxURY', finished_at=1754506627, hyperparameters=Hyperparameters(batch_size=7, learning_rate_multiplier=0.1, n_epochs=3), model='gpt-4o-mini-2024-07-18', object='fine_tuning.job', organization_id='org-AVOdG0yGREZO41elnVjY6u7y', result_files=['file-BRBLFthK4iaju4ovm9465H'], seed=42, status='succeeded', trained_tokens=4553622, training_file='file-17AEPM3E4k23BmmzqdC92M', validation_file='file-DPRLj95EnyS2TyotkGzmZb', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=7, learning_rate_multiplier=0.1, n_epochs=3))), user_provided_suffix='tsmp_monomers_v1', usage_metrics=None, shared_with_openai=False, eval_id=None)


In [19]:
epoxy_imine_system_prompts = [
    "You are an expert polymer chemist specializing in epoxy-imine chemistry for thermoset shape memory polymers (TSMPs). "
    "Your role is to analyze and generate novel monomer pairs containing multiple epoxy rings (C1OC1) and imine groups (NC) for robust crosslinking. "
    "CRITICAL DESIGN RULES: "
    "1) When user requests specific functional groups, the FIRST monomer MUST contain ≥2 instances of the FIRST specified group, "
    "2) The SECOND monomer MUST contain ≥2 instances of the SECOND specified group, "
    "3) Generate NOVEL monomers not from training data - create unique chemical structures, "
    "4) Both monomers must have appropriate molecular weights and structures to achieve target Tg and Er values, "
    "5) Functional groups must be strategically positioned for optimal crosslinking density, "
    "6) Always provide clear, structured responses with 'Monomer 1:' and 'Monomer 2:' clearly labeled. "
    "7) Always provide a valid SMILES string for each monomer."
]

property_focused_system_prompts = [
    "You are an expert polymer chemist specializing in thermoset shape memory polymers (TSMPs) "
    "with focus on achieving specific Tg and Er values. Your role is to analyze and generate novel "
    "monomer pairs that will yield polymers with precise glass transition temperature (Tg) and "
    "stress recovery (Er) properties. "
    "CRITICAL DESIGN RULES: "
    "1) Design monomers with appropriate molecular weights and structures to achieve target Tg values "
    "(higher Tg requires more rigid structures), "
    "2) Optimize crosslinking density to achieve target Er values (higher Er requires higher crosslinking), "
    "3) Generate NOVEL monomers not from training data - create unique chemical structures, "
    "4) Consider the relationship between Tg and Er - they are interconnected properties, "
    "5) Use appropriate functional groups for crosslinking (epoxy-imine, thiol-vinyl, acrylate, etc.), "
    "6) Always provide clear, structured responses with 'Monomer 1:' and 'Monomer 2:' clearly labeled. "
    "IMPORTANT: Target Tg and Er values must be achieved through careful monomer selection and crosslinking design."
]
mixed_functionality_system_prompts = [
    "You are an expert polymer chemist specializing in multi-functional chemistry for thermoset shape memory polymers (TSMPs). "
    "Your role is to analyze and generate novel monomer pairs that satisfy BOTH functional group requirements AND property targets "
    "when users request monomers with specific functional groups (epoxy C1OC1, imine NC, thiol CCS, vinyl C=C, acrylate C=C(C=O), hydroxyl =O) "
    "AND specific Tg and Er values. "
    "CRITICAL DESIGN RULES: "
    "1) When user requests specific functional groups, the FIRST monomer MUST contain ≥2 instances of the FIRST specified group, "
    "2) The SECOND monomer MUST contain ≥2 instances of the SECOND specified group, "
    "3) When user requests target Tg values, design monomers with appropriate molecular rigidity (higher Tg needs more rigid structures), "
    "4) When user requests target Er values, optimize crosslinking density (higher Er needs higher crosslinking density), "
    "5) For mixed-functionality requests, distribute complementary groups strategically across both monomers, "
    "6) Generate NOVEL monomers not from training data - create unique chemical structures, "
    "7) Balance functional group placement with molecular structure to achieve BOTH group requirements AND property targets, "
    "8) Always provide clear, structured responses with 'Monomer 1:' and 'Monomer 2:' clearly labeled. "
    "9) Always provide a valid SMILES string for each monomer. "
]

In [21]:
system_prompt = "You are a polymer expert specializing in thermoset shape memory polymers. Your role is to analyze and create monomer pairs with excellent thermal stability and mechanical strength."
user_message_1 =['I want to make a thermoset shape memory polymer','Please suggest me some TSMPs']
user_message_2 =['Please focus on property based monomer pairs','Please focus on group based monomer pairs', "both"]
proeprty_specific_message = ["Please give me some TSMP with Tg = 100C and Er= 150MPa","Please generate some TSMP with Tg = 50C and Er= 100Mpa"]
group_specific_message = ["Please give me some TSMP with epoxy(C1OC1) groups in monomer 1 and imine(NC) groups in monomer 2","Please generate some TSMP with vinyl(C=C) groups in monomer 1 and thiol(CCS) groups in monomer 2"]
mixed_specific_message = ["Please give me some TSMP with Tg = 100C and Er= 150MPa and vinyl(C=C) groups in monomer 1 and vinyl(C=C) groups in monomer 2","Please generate some TSMP with Tg = 50C and Er= 100Mpa and Thiol(CCS) groups in monomer 1 and vinyl(C=C) groups in monomer 2"]


In [24]:
messages=[]
messages.append({"role":"system","content":mixed_functionality_system_prompts[0]})
def generate_new_TSMP(role,prompt_content, isFinalQuery=False):
    propmt={"role":role, "content": prompt_content}
    messages.append(propmt)
    if not isFinalQuery:
        completion = client.chat.completions.create(
            model='ft:gpt-4o-mini-2024-07-18:personal:tsmp-monomers-v1:C1dbxURY',
            messages=messages
        )
        result = completion.choices[0].message.content
    else:
        completion = client.chat.completions.create(
            model='ft:gpt-4o-mini-2024-07-18:personal:tsmp-monomers-v1:C1dbxURY',
            messages=messages,
            temperature=1.0,
            max_tokens=200,
            n=2
        )
        result= completion
       
        
   
    messages.append({"role":'assistant', "content": result})
    
    # Print the response from the assistant
    return result, messages


replies_0, messages_0 = generate_new_TSMP('user',user_message_1[0], isFinalQuery=False)
print("User: ",user_message_1[0])
print("Assistant: ",replies_0)
replies_1, messages_1 = generate_new_TSMP('user',user_message_2[2], isFinalQuery=False)
print("User: ",user_message_2[2])
print("Assistant: ",replies_1)
replies_2, messages_2 = generate_new_TSMP('user',mixed_specific_message[0], isFinalQuery=True)
print("User: ",mixed_specific_message[0])
print("Assistant: Two TSMPs for you ( as I told model to generate two samples per query) ")
for i in range(2):
    print("Sample : :",i+1, replies_2.choices[i].message.content)
    

User:  I want to make a thermoset shape memory polymer
Assistant:  What's more important for your TSMP - certain functional groups or specific properties?
User:  both
Assistant:  What's your priority - working with specific chemical groups like epoxy (C1OC1), imine (NC), vinyl (C=C), or targeting specific properties like Tg and Er?
User:  Please give me some TSMP with Tg = 100C and Er= 150MPa and vinyl(C=C) groups in monomer 1 and vinyl(C=C) groups in monomer 2
Assistant: Two TSMPs for you ( as I told model to generate two samples per query) 
Sample : : 1 Here's a valid monomer pair designed with vinyl groups and optimized for Tg ≈ 100°C and Er ≈ 150MPa:
Monomer 1: C=CCCCC(=O)OCCn(CCn(CCOC(=O)CCC=C)CCOC(=O)CC=C)CCOC(=O)CCCC(C)=OCC=C
Monomer 2: C=CCc1cccc(CC=C)cc1
Sample : : 2 To meet the criteria of Tg = 100°C, Er = 150MPa, and C=C groups, I designed these monomers:
Monomer 1: C=CCOC(CC)(CC)CC(C)=O
Monomer 2: C=C(CCS(=O)(=O)OC(CC)(C)C)OCCOCCOCCOCCOCCOCCOCCOCCOCCOCCOCCOCCOCCOCCOCCOCCOCC

In [28]:

thiol_ene_system_prompts = [
    "You are an expert polymer chemist specializing in thiol-vinyl chemistry for thermoset shape memory polymers (TSMPs). "
    "Your role is to analyze and generate novel monomer pairs containing multiple thiol groups (CCS) and vinyl groups (C=C) for robust crosslinking. "
    "CRITICAL DESIGN RULES: "
    "1) When user requests specific functional groups, the FIRST monomer MUST contain ≥2 instances of the FIRST specified group, "
    "2) The SECOND monomer MUST contain ≥2 instances of the SECOND specified group, "
    "3) Generate NOVEL monomers not from training data - create unique chemical structures, "
    "4) Both monomers must have appropriate molecular weights and structures to achieve target Tg and Er values, "
    "5) Functional groups must be strategically positioned for optimal crosslinking density, "
    "6) Always provide clear, structured responses with 'Monomer 1:' and 'Monomer 2:' clearly labeled. "
    "7) Always provide a valid SMILES string for each monomer."
]
messages=[]
messages.append({"role":"system","content":thiol_ene_system_prompts[0]})
def generate_new_TSMP(role,prompt_content, isFinalQuery=False):
    propmt={"role":role, "content": prompt_content}
    messages.append(propmt)
    if not isFinalQuery:
        completion = client.chat.completions.create(
            model='ft:gpt-4o-mini-2024-07-18:personal:tsmp-monomers-v1:C1dbxURY',
            messages=messages
        )
        result = completion.choices[0].message.content
    else:
        completion = client.chat.completions.create(
            model='ft:gpt-4o-mini-2024-07-18:personal:tsmp-monomers-v1:C1dbxURY',
            messages=messages,
            temperature=0.6,
            max_tokens=200,
            n=4
        )
        result= completion
       
        
   
    messages.append({"role":'assistant', "content": result})
    
    # Print the response from the assistant
    return result, messages

replies, messages = generate_new_TSMP('user',group_specific_message[1], isFinalQuery=True)
print("User: ",group_specific_message[1])
print("Assistant: Two TSMPs for you ( as I told model to generate two samples per query) ")
for i in range(4):
    print("Sample : :",i+1, replies.choices[i].message.content)

User:  Please generate some TSMP with vinyl(C=C) groups in monomer 1 and thiol(CCS) groups in monomer 2
Assistant: Two TSMPs for you ( as I told model to generate two samples per query) 
Sample : : 1 Here's a TSMP that includes the requested functional groups:
Monomer 1 (contains C=C): C=C(C)C(=C)OCOCCOC(=O)C(=C)C
Monomer 2 (contains CCS): CCCC(=O)OCCCCC
Sample : : 2 Sure, here's a group-compliant monomer pair:
Monomer 1: C=C(C)C(=C)OCOCCOC(=O)C(=C)C
Monomer 2: CCCCCCCCCCCSCCCCCCCCCCC
Sample : : 3 Here is a TSMP that includes the requested functional groups:
Monomer 1 (vinyl): C=CC(=O)OCCn1c(=O)n(CCOC(=O)C=C)c(=O)n(CCOC(=O)C=C)c(=O)n1CCOC(=O)C=C
Monomer 2 (CCS): C=C(C)C(=O)OCCS(=O)(=O)NCC1(C)CC(C1)N(CCS(=O)(=O)OCC(=O)C(=C)C)CCS(=O)(=O)OCC(=O)C(=C)C
Sample : : 4 Here's a TSMP that includes the requested functional groups:
Monomer 1 (C=C): C=C(C)C(=C)OCOCCOC(=O)C(=C)C
Monomer 2 (CCS): CCCC(=O)OCCCCC


In [29]:
print(thiol_ene_system_prompts[0])

You are an expert polymer chemist specializing in thiol-vinyl chemistry for thermoset shape memory polymers (TSMPs). Your role is to analyze and generate novel monomer pairs containing multiple thiol groups (CCS) and vinyl groups (C=C) for robust crosslinking. CRITICAL DESIGN RULES: 1) When user requests specific functional groups, the FIRST monomer MUST contain ≥2 instances of the FIRST specified group, 2) The SECOND monomer MUST contain ≥2 instances of the SECOND specified group, 3) Generate NOVEL monomers not from training data - create unique chemical structures, 4) Both monomers must have appropriate molecular weights and structures to achieve target Tg and Er values, 5) Functional groups must be strategically positioned for optimal crosslinking density, 6) Always provide clear, structured responses with 'Monomer 1:' and 'Monomer 2:' clearly labeled. 7) Always provide a valid SMILES string for each monomer.
